In [1]:
!pip install -U transformers accelerate torchaudio pandas tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 135.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 154.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26

In [2]:
import torch
import pandas as pd
from transformers import pipeline
from tqdm import tqdm


In [3]:
device = 0 if torch.cuda.is_available() else -1

asr = pipeline(
    task="automatic-speech-recognition",
    model="vasista22/whisper-tamil-large-v2",
    # chunk_length_s=30,
    device=device,
    torch_dtype=torch.float16
)

# ✅ IMPORTANT: force Tamil via decoder IDs (NOT via generate())
asr.model.config.forced_decoder_ids = (
    asr.tokenizer.get_decoder_prompt_ids(
        language="en",
        task="transcribe"
    )
)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
asr.model.eval()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/6.17G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.17G [00:00<?, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 1280, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(1280, 1280, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 1280)
      (layers): ModuleList(
        (0-31): 32 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=1280, out_features=1280, bias=False)
            (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
            (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
            (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1280, out_features=5120, bias=True)
          (fc2): Linear(in_features=5120, out_features=1280, bias=Tru

In [5]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
INPUT_CSV = "/content/drive/MyDrive/output_clean_cs_data.csv"
OUTPUT_CSV = "/content/drive/MyDrive/output_metrics_set_to_default_large_v2.csv"

df = pd.read_csv(INPUT_CSV)
df.columns = df.columns.str.strip()

results = []


In [ ]:
from datasets import Dataset

# Create HF dataset
dataset = Dataset.from_pandas(df)

def transcribe_batch(batch):
    outputs = asr(
        batch["audio_wav_path"],
        batch_size=16   # 🔥 IMPORTANT: GPU batch
    )
    return {
        "transcription": [o["text"] for o in outputs]
    }

# Run batched transcription
dataset = dataset.map(
    transcribe_batch,
    batched=True,
    batch_size=16
)

# Convert back to DataFrame
out_df = dataset.to_pandas()

# Save CSV
out_df[["script_text", "audio_wav_path", "transcription"]].to_csv(
    OUTPUT_CSV,
    index=False
)

print("✅ Transcription completed with GPU batching!")


Map:   0%|          | 0/3685 [00:00<?, ? examples/s]

`return_token_timestamps` is deprecated for WhisperFeatureExtractor and will be removed in Transformers v5. Use `return_attention_mask` instead, as the number of frames can be inferred from it.
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


✅ Transcription completed with GPU batching!


In [6]:
#only first 16 files for english
import pandas as pd
from datasets import Dataset

INPUT_CSV = "/content/drive/MyDrive/output_clean_cs_data.csv"
OUTPUT_CSV = "/content/drive/MyDrive/output_metrics_set_to_english_large_v2.csv"

# Read CSV and take only the first 16 rows
df = pd.read_csv(INPUT_CSV)
df.columns = df.columns.str.strip()
df = df.head(16)  # 🔥 Only first 16 rows

# Create HF dataset
dataset = Dataset.from_pandas(df)

def transcribe_batch(batch):
    outputs = asr(
        batch["audio_wav_path"],
        batch_size=16   # GPU batch
    )
    return {
        "transcription": [o["text"] for o in outputs]
    }

# Run batched transcription
dataset = dataset.map(
    transcribe_batch,
    batched=True,
    batch_size=16
)

# Convert back to DataFrame
out_df = dataset.to_pandas()

# Save CSV
out_df[["script_text", "audio_wav_path", "transcription"]].to_csv(
    OUTPUT_CSV,
    index=False
)

print("✅ Transcription completed for first 16 rows only!")


Map:   0%|          | 0/16 [00:00<?, ? examples/s]

`return_token_timestamps` is deprecated for WhisperFeatureExtractor and will be removed in Transformers v5. Use `return_attention_mask` instead, as the number of frames can be inferred from it.
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.


✅ Transcription completed for first 16 rows only!


In [ ]:
#This code block not needed
for idx, row in tqdm(df.iterrows(), total=len(df)):
    try:
        audio_path = row["audio_wav_path"]

        result = asr(audio_path)
        tamil_text = result["text"]

        results.append({
            "original_text": row.get("script_text", ""),
            "audio": audio_path,
            "transcription": tamil_text
        })

    except Exception as e:
        print(f"❌ Error at row {idx}: {e}")

out_df = pd.DataFrame(results)
out_df.to_csv(OUTPUT_CSV, index=False)

print("✅ Transcription completed!")


  1%|          | 34/3685 [10:43<19:12:20, 18.94s/it]


KeyboardInterrupt: 

In [ ]:
!pip install jiwer regex pandas


In [ ]:
import pandas as pd

INPUT_CSV = "/content/drive/MyDrive/vasista_zero_shot/output_clean_cs_data.csv"
OUTPUT_CSV = "/content/drive/MyDrive/vasista_zero_shot/output_metrics.csv"

df = pd.read_csv(INPUT_CSV)
df.columns = df.columns.str.strip()

print("Rows:", len(df))
df.head()


In [ ]:
from jiwer import wer


In [ ]:
def compute_wer(reference, hypothesis):
    if not isinstance(reference, str) or not isinstance(hypothesis, str):
        return None
    return wer(reference.lower(), hypothesis.lower())


In [ ]:
import re

tamil_pattern = re.compile(r"[\u0B80-\u0BFF]")
english_pattern = re.compile(r"[A-Za-z]")


In [ ]:
def word_language(word):
    if tamil_pattern.search(word):
        return "ta"
    elif english_pattern.search(word):
        return "en"
    else:
        return "other"


In [ ]:
def compute_cmi(sentence, wN=0.5, wA=0.5):
    if not isinstance(sentence, str):
        return None

    words = sentence.split()
    if len(words) == 0:
        return 0.0

    langs = [word_language(w) for w in words]

    N = len(words)
    N_ta = langs.count("ta")
    N_en = langs.count("en")

    # Count code-switching points
    alpha = 0
    for i in range(1, len(langs)):
        if langs[i] != langs[i-1] and langs[i] in ["ta", "en"] and langs[i-1] in ["ta", "en"]:
            alpha += 1

    cmi = (wN * min(N_ta, N_en) / N) + (wA * alpha / N)
    return cmi


In [ ]:
wer_list = []
cmi_list = []

for _, row in df.iterrows():
    ref = row["script_text"]
    hyp = row["transcription"]

    wer_val = compute_wer(ref, hyp)
    cmi_val = compute_cmi(ref)  # CMI computed on ground truth

    wer_list.append(wer_val)
    cmi_list.append(cmi_val)


In [ ]:
df["WER"] = wer_list
df["CMI"] = cmi_list


In [ ]:
avg_wer = df["WER"].dropna().mean()
avg_cmi = df["CMI"].dropna().mean()

print(f"📌 Average WER : {avg_wer:.4f}")
print(f"📌 Average CMI : {avg_cmi:.4f}")
